# Vocoder Comparison: Python mlx-audio vs Rust voicers

Same phonemes, same model weights, same voice — which produces intelligible audio?

If Python sounds fine and Rust doesn't, the bug is in our Rust vocoder (iSTFT, overlap-add, weight loading, etc).
If both sound bad, it's the model itself or an MLX framework issue.

In [1]:
import subprocess, json, os, tempfile
import numpy as np
import whisper

VOICERS_CLI = "./target/release/voicers-cli"

# Build release binary
result = subprocess.run(
    ["cargo", "build", "--release", "-p", "voicers-cli"], capture_output=True, text=True
)
print("voicers-cli:", "ready" if result.returncode == 0 else result.stderr[-200:])

# Load Whisper
whisper_model = whisper.load_model("base")
print("Whisper: ready")

FileNotFoundError: [Errno 2] No such file or directory: 'cargo'

In [2]:
import subprocess, json, os, tempfile
import numpy as np

# Full paths since notebook env may not have cargo in PATH
CARGO = os.path.expanduser("~/.cargo/bin/cargo")
VOICERS_CLI = os.path.abspath("./target/release/voicers-cli")

# Build release binary
result = subprocess.run(
    [CARGO, "build", "--release", "-p", "voicers-cli"], capture_output=True, text=True
)
print("voicers-cli:", "ready" if result.returncode == 0 else result.stderr[-200:])

import whisper

whisper_model = whisper.load_model("base")
print("Whisper: ready")

voicers-cli: ready
Whisper: ready


In [3]:
# Set up Python mlx-audio Kokoro pipeline
from mlx_audio.tts.models.kokoro.pipeline import KokoroPipeline
import soundfile as sf

pipeline = KokoroPipeline(lang_code="a")
print(f"Python mlx-audio pipeline ready, voice: af_heart")

ModuleNotFoundError: No module named 'misaki'

In [1]:
import subprocess, json, os, tempfile
import numpy as np

CARGO = os.path.expanduser("~/.cargo/bin/cargo")
VOICERS_CLI = os.path.abspath("./target/release/voicers-cli")

# Build
result = subprocess.run(
    [CARGO, "build", "--release", "-p", "voicers-cli"], capture_output=True, text=True
)
print("voicers-cli:", "ready" if result.returncode == 0 else result.stderr[-200:])

import whisper

whisper_model = whisper.load_model("base")
print("Whisper: ready")

voicers-cli: ready
Whisper: ready


In [2]:
# Set up Python mlx-audio Kokoro pipeline
from mlx_audio.tts.models.kokoro.pipeline import KokoroPipeline
import soundfile as sf

pipeline = KokoroPipeline(lang_code="a")
print(f"Python mlx-audio Kokoro pipeline ready")

TypeError: KokoroPipeline.__init__() missing 2 required positional arguments: 'model' and 'repo_id'

In [3]:
# Use mlx-audio's generate CLI and our voicers CLI to produce audio from the same text
# Then transcribe both with Whisper

import soundfile as sf


def generate_python_audio(text, voice="af_heart", output_path=None):
    """Generate audio using Python mlx-audio Kokoro."""
    if output_path is None:
        output_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [
            "python3",
            "-m",
            "mlx_audio.tts.generate",
            "--model",
            "prince-canuma/Kokoro-82M",
            "--text",
            text,
            "--voice",
            voice,
            "--output",
            output_path,
        ],
        capture_output=True,
        text=True,
        timeout=120,
    )
    if r.returncode != 0:
        print(f"Python gen failed: {r.stderr[-300:]}")
        return None
    return output_path


def generate_rust_audio(text, voice="af_heart", output_path=None):
    """Generate audio using Rust voicers CLI."""
    if output_path is None:
        output_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [VOICERS_CLI, "--text", text, "--voice", voice, "--output", output_path],
        capture_output=True,
        text=True,
        timeout=120,
    )
    if r.returncode != 0:
        print(f"Rust gen failed: {r.stderr[-300:]}")
        return None
    # Extract phonemes from stderr
    phonemes = ""
    for line in r.stderr.splitlines():
        if line.strip().startswith("chunk"):
            phonemes += line.split(":", 1)[1].strip() + " "
    return output_path, phonemes.strip()


def transcribe(wav_path):
    """Transcribe audio with Whisper."""
    result = whisper_model.transcribe(wav_path, language="en")
    return result["text"].strip()


# Quick smoke test
print("Testing Python mlx-audio...")
py_path = generate_python_audio("Hello world")
print(f"  Generated: {py_path}")
if py_path:
    py_whisper = transcribe(py_path)
    print(f"  Whisper heard: {py_whisper}")
    os.unlink(py_path)

Testing Python mlx-audio...
Python gen failed: /Applications/Xcode.app/Contents/Developer/usr/bin/python3: E
rror while finding module specification for 'mlx_audio.tts.generate' (ModuleNotF
oundError: No module named 'mlx_audio')

  Generated: None


In [4]:
# Generate Python audio directly in this notebook since we have mlx-audio installed here
import mlx.core as mx
from mlx_audio.tts.models.kokoro.kokoro import (
    Model,
    ModelConfig,
    sanitize,
    sanitize_lstm_weights,
)
from mlx_audio.tts.models.kokoro.voice import load_voice_tensor
from huggingface_hub import snapshot_download
import mlx.nn as nn
import json
from pathlib import Path

# Load model
repo_id = "prince-canuma/Kokoro-82M"
model_path = Path(snapshot_download(repo_id))
print(f"Model path: {model_path}")

with open(model_path / "config.json") as f:
    config_data = json.load(f)

config = ModelConfig.from_dict(config_data)
model = Model(config)

# Load weights
weights = mx.load(str(model_path / "model.safetensors"))
sanitized = sanitize(weights, config)
model.load_weights(list(sanitized.items()), strict=False)
mx.eval(model.parameters())
print("Model loaded")

# Load voice
voice = load_voice_tensor(repo_id, "af_heart")
print(f"Voice loaded: {voice.shape}")

ImportError: cannot import name 'sanitize' from 'mlx_audio.tts.models.kokoro.kokoro' (/Users/kylekelley/Library/Caches/runt/inline-envs/578aa62ad89c9c27/lib/python3.12/site-packages/mlx_audio/tts/models/kokoro/kokoro.py)

In [5]:
# Let's see what mlx-audio's Model exposes — the installed version may differ from colombo-v2 source
from mlx_audio.tts.models.kokoro.kokoro import Model, ModelConfig

print(dir(Model))

# Use the high-level generate API instead of manually loading
# First, let's see what the Model.generate method signature looks like
import inspect

sig = inspect.signature(Model.generate)
print(f"\nModel.generate signature: {sig}")

['Output', 'REPO_ID', '__annotations__', '__call__', '__class__', '__class_getit
em__', '__contains__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__d
oc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__g
etitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__'
, '__ior__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__',
'__new__', '__or__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__',
'__ror__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshoo
k__', '__weakref__', '_extra_repr', '_get_pipeline', '_set_training_mode', '_val
idate_keys', 'apply', 'apply_to_modules', 'children', 'clear', 'copy', 'eval', '
filter_and_map', 'freeze', 'fromkeys', 'generate', 'get', 'is_module', 'items',
'keys', 'leaf_modules', 'load_weights', 'modules', 'named_modules', 'parameters'
, 'pop', 'popitem', 'sample_rate', 'sanitize', 'save_weights', 'set_dtype', 'set
default', 'state', 'train', 'tr

In [6]:
# Use the high-level API — Model handles its own loading
from mlx_audio.tts.utils import load as load_tts

py_model, py_processor = load_tts("prince-canuma/Kokoro-82M")
print(f"Python model loaded, sample_rate={py_model.sample_rate}")

Fetching 63 files:   0%|          | 0/63 [00:00<?, ?it/s]

ValueError: too many values to unpack (expected 2)

In [7]:
# Check what load returns
from mlx_audio.tts.utils import load as load_tts

result = load_tts("prince-canuma/Kokoro-82M")
print(type(result))
if isinstance(result, tuple):
    print(f"Tuple length: {len(result)}")
    for i, r in enumerate(result):
        print(f"  [{i}] {type(r)}")
else:
    py_model = result
    print(f"Single object: {type(py_model)}")

Fetching 63 files:   0%|          | 0/63 [00:00<?, ?it/s]

<class 'mlx_audio.tts.models.kokoro.kokoro.Model'>
Single object: <class 'mlx_audio.tts.models.kokoro.kokoro.Model'>


In [8]:
# Now we have the Python model. Let's generate audio!
import soundfile as sf

py_model = result  # from previous cell


def python_generate(text, voice="af_heart"):
    """Generate audio using Python mlx-audio and return wav path."""
    wav_path = tempfile.mktemp(suffix=".wav")
    for gen_result in py_model.generate(text, voice=voice, lang_code="a"):
        audio = np.array(gen_result.audio)
        sf.write(wav_path, audio, gen_result.sample_rate)
        return wav_path  # just take first segment
    return None


def rust_generate(text, voice="af_heart"):
    """Generate audio using Rust voicers and return (wav_path, phonemes)."""
    wav_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [VOICERS_CLI, "--text", text, "--voice", voice, "--output", wav_path],
        capture_output=True,
        text=True,
        timeout=120,
    )
    phonemes = ""
    for line in r.stderr.splitlines():
        if line.strip().startswith("chunk"):
            phonemes += line.split(":", 1)[1].strip() + " "
    return wav_path, phonemes.strip()


# Smoke test
print("=== Python mlx-audio ===")
py_wav = python_generate("Hello world")
py_text = transcribe(py_wav)
print(f"  Whisper heard: {py_text}")

print("\n=== Rust voicers ===")
rust_wav, rust_ph = rust_generate("Hello world")
rust_text = transcribe(rust_wav)
print(f"  Phonemes: {rust_ph}")
print(f"  Whisper heard: {rust_text}")

# Cleanup
os.unlink(py_wav)
os.unlink(rust_wav)

=== Python mlx-audio ===
Creating new KokoroPipeline for language: a


/Users/kylekelley/Library/Caches/runt/inline-envs/578aa62ad89c9c27/bin/python: N
o module named pip


SystemExit: 1

/Users/kylekelley/Library/Caches/runt/inline-envs/578aa62ad89c9c27/lib/python3.1
2/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use
 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import subprocess, os, tempfile
import numpy as np

CARGO = os.path.expanduser("~/.cargo/bin/cargo")
VOICERS_CLI = os.path.abspath("./target/release/voicers-cli")

# Verify spacy model is available
import spacy

nlp = spacy.load("en_core_web_sm")
print("spaCy en_core_web_sm: ready")

# Load Whisper
import whisper

whisper_model = whisper.load_model("base")
print("Whisper: ready")

# Load Python mlx-audio model
from mlx_audio.tts.utils import load as load_tts

py_model = load_tts("prince-canuma/Kokoro-82M")
print(f"Python Kokoro: ready (sample_rate={py_model.sample_rate})")

spaCy en_core_web_sm: ready
Whisper: ready


Fetching 63 files:   0%|          | 0/63 [00:00<?, ?it/s]

Python Kokoro: ready (sample_rate=24000)


In [2]:
import soundfile as sf


def python_generate(text, voice="af_heart"):
    """Generate audio using Python mlx-audio Kokoro."""
    wav_path = tempfile.mktemp(suffix=".wav")
    for gen_result in py_model.generate(text, voice=voice, lang_code="a"):
        audio = np.array(gen_result.audio)
        sf.write(wav_path, audio, gen_result.sample_rate)
        return wav_path
    return None


def rust_generate(text, voice="af_heart"):
    """Generate audio using Rust voicers."""
    wav_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [VOICERS_CLI, "--text", text, "--voice", voice, "--output", wav_path],
        capture_output=True,
        text=True,
        timeout=120,
    )
    phonemes = ""
    for line in r.stderr.splitlines():
        if line.strip().startswith("chunk"):
            phonemes += line.split(":", 1)[1].strip() + " "
    return wav_path, phonemes.strip()


def transcribe(wav_path):
    result = whisper_model.transcribe(wav_path, language="en")
    return result["text"].strip()


# Smoke test both
print("=== Python mlx-audio ===")
py_wav = python_generate("Hello world")
print(f"  Whisper heard: {transcribe(py_wav)}")
os.unlink(py_wav)

print("\n=== Rust voicers ===")
rust_wav, rust_ph = rust_generate("Hello world")
print(f"  Phonemes: {rust_ph}")
print(f"  Whisper heard: {transcribe(rust_wav)}")
os.unlink(rust_wav)

=== Python mlx-audio ===
Creating new KokoroPipeline for language: a


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


=== Python mlx-audio ===
Creating new KokoroPipeline for language: a
  Whisper heard: Hello world.

=== Rust voicers ===
  Phonemes: həlˈO wˈɜɹld


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


=== Python mlx-audio ===
Creating new KokoroPipeline for language: a
  Whisper heard: Hello world.

=== Rust voicers ===
  Phonemes: həlˈO wˈɜɹld
  Whisper heard: Hello, where are you?


In [3]:
# Full A/B comparison
test_phrases = [
    "Hello world",
    "Good morning",
    "The quick brown fox",
    "How are you doing today?",
    "She sells seashells by the seashore.",
    "I read a book yesterday.",
    "The dog is running in the park.",
]

print(f"{'Input':<40} {'Python Whisper':<30} {'Rust Whisper':<30}")
print("=" * 100)

for phrase in test_phrases:
    # Python
    py_wav = python_generate(phrase)
    py_heard = transcribe(py_wav) if py_wav else "FAILED"
    os.unlink(py_wav) if py_wav else None

    # Rust
    rust_wav, rust_ph = rust_generate(phrase)
    rust_heard = transcribe(rust_wav)
    os.unlink(rust_wav)

    py_ok = "OK" if phrase.lower().rstrip(".,!?") in py_heard.lower() else ""
    rust_ok = "OK" if phrase.lower().rstrip(".,!?") in rust_heard.lower() else ""

    print(f"{phrase:<40} {py_heard[:28]:<30} {rust_heard[:28]:<30}")

Input                                    Python Whisper                 Rust Whi
sper


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Input                                    Python Whisper                 Rust Whi
sper
Hello world                              Hello world.                   Hello, w
here? Holds that.


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python Whisper                 Rust Whi
sper
Hello world                              Hello world.                   Hello, w
here? Holds that.
Good morning                             Good morning.                  Good mor
ning, hey


...

Input                                    Python Whisper                 Rust Whi
sper
Hello world                              Hello world.                   Hello, w
here? Holds that.
Good morning                             Good morning.                  Good mor
ning, hey
The quick brown fox                      The quick brown fox.           The cool
, very evil!
How are you doing today?                 How are you doing today?       How art
you doing to say
She sells seashells by the seashore.     She sells seashells by the s   She sell
s a sea shells by th


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python Whisper                 Rust Whi
sper
Hello world                              Hello world.                   Hello, w
here? Holds that.
Good morning                             Good morning.                  Good mor
ning, hey
The quick brown fox                      The quick brown fox.           The cool
, very evil!
How are you doing today?                 How are you doing today?       How art
you doing to say
She sells seashells by the seashore.     She sells seashells by the s   She sell
s a sea shells by th
I read a book yesterday.                 I read a book yesterday.       I read a
 book, yes, to Herds


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python Whisper                 Rust Whi
sper
Hello world                              Hello world.                   Hello, w
here? Holds that.
Good morning                             Good morning.                  Good mor
ning, hey
The quick brown fox                      The quick brown fox.           The cool
, very evil!
How are you doing today?                 How are you doing today?       How art
you doing to say
She sells seashells by the seashore.     She sells seashells by the s   She sell
s a sea shells by th
I read a book yesterday.                 I read a book yesterday.       I read a
 book, yes, to Herds
The dog is running in the park.          The dog is running in the pa   The Dizd
og is as running int


## Results: Python mlx-audio vs Rust voicers

| Input | Python (Whisper) | Rust (Whisper) |
|-------|-----------------|----------------|
| Hello world | Hello world. | Hello, where? Holds that. |
| Good morning | Good morning. | Good morning, hey |
| The quick brown fox | The quick brown fox. | The cool, very evil! |
| How are you doing today? | How are you doing today? | How art you doing to say |
| She sells seashells... | She sells seashells by the s... | She sells a sea shells by th... |
| I read a book yesterday. | I read a book yesterday. | I read a book, yes, to Herds |
| The dog is running... | The dog is running in the pa... | The Dizdog is as running int... |

**Python: 7/7 correct. Rust: 0/7 correct.**

The Python mlx-audio pipeline produces perfect audio from the same model. Our Rust vocoder is the problem. The issue is somewhere in:
1. **iSTFT reconstruction** (overlap-add, window normalization)
2. **Weight loading/sanitization** (some weights may still be misaligned)
3. **Forward pass numerics** (intermediate computation differences)

Next: dump intermediate tensors from both Python and Rust to find where they diverge.

In [1]:
import subprocess, os, tempfile
import numpy as np
import soundfile as sf

CARGO = os.path.expanduser("~/.cargo/bin/cargo")
VOICERS_CLI = os.path.abspath("./target/release/voicers-cli")

import whisper

whisper_model = whisper.load_model("base")
print("Whisper: ready")

from mlx_audio.tts.utils import load as load_tts

py_model = load_tts("prince-canuma/Kokoro-82M")
print(f"Python Kokoro: ready")


def transcribe(wav_path):
    return whisper_model.transcribe(wav_path, language="en")["text"].strip()


def python_generate(text, voice="af_heart"):
    wav_path = tempfile.mktemp(suffix=".wav")
    for gen_result in py_model.generate(text, voice=voice, lang_code="a"):
        audio = np.array(gen_result.audio)
        sf.write(wav_path, audio, gen_result.sample_rate)
        return wav_path
    return None


def rust_generate(text, voice="af_heart"):
    wav_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [VOICERS_CLI, "--text", text, "--voice", voice, "--output", wav_path],
        capture_output=True,
        text=True,
        timeout=120,
    )
    phonemes = ""
    for line in r.stderr.splitlines():
        if line.strip().startswith("chunk"):
            phonemes += line.split(":", 1)[1].strip() + " "
    return wav_path, phonemes.strip()


print("All ready.")

Whisper: ready


Fetching 63 files:   0%|          | 0/63 [00:00<?, ?it/s]

Python Kokoro: ready
All ready.


In [2]:
# The specific test phrase
test_text = "Now the spaCy tokenizer uses uv run with spacy. Zero setup needed beyond having uv installed. And we get proper POS tagging."

# Python reference
print("=== Python mlx-audio ===")
py_wav = python_generate(test_text)
py_heard = transcribe(py_wav)
print(f"  Whisper: {py_heard}")

# Rust voicers (with iSTFT fix)
print("\n=== Rust voicers ===")
rust_wav, rust_ph = rust_generate(test_text)
rust_heard = transcribe(rust_wav)
print(f"  Phonemes: {rust_ph}")
print(f"  Whisper:  {rust_heard}")

os.unlink(py_wav)
os.unlink(rust_wav)

=== Python mlx-audio ===
Creating new KokoroPipeline for language: a


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


=== Python mlx-audio ===
Creating new KokoroPipeline for language: a
  Whisper: Now the SpawSci tokenizer uses UV run with Spacy. Zero setup needed b
eyond having UV installed, and we get proper POS tagging.

=== Rust voicers ===


FileNotFoundError: [Errno 2] No such file or directory: '/Users/kylekelley/conductor/workspaces/voicers/algiers/notebooks/target/release/voicers-cli'

In [3]:
# Fix path - use absolute path from repo root
import os

REPO_ROOT = os.path.dirname(os.getcwd())  # parent of notebooks/
VOICERS_CLI = os.path.join(REPO_ROOT, "target/release/voicers-cli")
print(f"CLI: {VOICERS_CLI}")
print(f"Exists: {os.path.exists(VOICERS_CLI)}")


def rust_generate(text, voice="af_heart"):
    wav_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [VOICERS_CLI, "--text", text, "--voice", voice, "--output", wav_path],
        capture_output=True,
        text=True,
        timeout=120,
    )
    phonemes = ""
    for line in r.stderr.splitlines():
        if line.strip().startswith("chunk"):
            phonemes += line.split(":", 1)[1].strip() + " "
    return wav_path, phonemes.strip()


# Now test Rust
print("\n=== Rust voicers (after iSTFT fix) ===")
rust_wav, rust_ph = rust_generate(test_text)
rust_heard = transcribe(rust_wav)
print(f"  Phonemes: {rust_ph}")
print(f"  Whisper:  {rust_heard}")
os.unlink(rust_wav)

CLI: /Users/kylekelley/conductor/workspaces/voicers/algiers/target/release/voice
rs-cli
Exists: True

=== Rust voicers (after iSTFT fix) ===


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


CLI: /Users/kylekelley/conductor/workspaces/voicers/algiers/target/release/voice
rs-cli
Exists: True

=== Rust voicers (after iSTFT fix) ===
  Phonemes: nˈW ðə spˈɑ sˈI tˈOkənˌIzəɹ jˈuzᵻz jˌuvˈi ɹˈʌn wɪð spˈAsi. zˈɪɹO sˈɛ
Tˌʌp nˈidᵻd bijˈɑnd hˈævɪŋ jˌuvˈi ɪnstˈɔld. ˌænd wi ɡɛt pɹˈɑpəɹ pˌiˌOˈɛs tˈæɡɪŋ.
  Whisper:  Now the spupa, cy, touger, canonizer uses this 2v run with spiff bas
e AC, zero set up, an idiot, pionned, having the uv insist alled, and we get a r
obber, p-a-o-s tagging.


In [4]:
# Full A/B comparison with the standard test phrases
test_phrases = [
    "Hello world",
    "Good morning",
    "The quick brown fox",
    "How are you doing today?",
    "She sells seashells by the seashore.",
    "I read a book yesterday.",
    "The dog is running in the park.",
]

print(f"{'Input':<40} {'Python':<35} {'Rust':<35}")
print("=" * 110)

py_score = 0
rust_score = 0

for phrase in test_phrases:
    py_wav = python_generate(phrase)
    py_heard = transcribe(py_wav) if py_wav else "FAILED"
    os.unlink(py_wav) if py_wav else None

    rust_wav, _ = rust_generate(phrase)
    rust_heard = transcribe(rust_wav)
    os.unlink(rust_wav)

    # Loose match: check if most words appear
    p_norm = phrase.lower().rstrip(".,!?")
    py_ok = p_norm in py_heard.lower().rstrip(".,!?")
    rust_ok = p_norm in rust_heard.lower().rstrip(".,!?")
    py_score += int(py_ok)
    rust_score += int(rust_ok)

    print(f"{phrase:<40} {py_heard[:33]:<35} {rust_heard[:33]:<35}")

print(
    f"\nPython: {py_score}/{len(test_phrases)}   Rust: {rust_score}/{len(test_phrases)}"
)

Input                                    Python                              Rus
t


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Input                                    Python                              Rus
t
Hello world                              Hello world.                        Hel
lo, we're a heraldsess.


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python                              Rus
t
Hello world                              Hello world.                        Hel
lo, we're a heraldsess.
Good morning                             Good morning.                       Goo
d. More anything.


...

Input                                    Python                              Rus
t
Hello world                              Hello world.                        Hel
lo, we're a heraldsess.
Good morning                             Good morning.                       Goo
d. More anything.
The quick brown fox                      The quick brown fox.                The
y...wicked brown fagus.
How are you doing today?                 How are you doing today?            How
 art you do wings to daddy?
She sells seashells by the seashore.     She sells seashells by the seasho   She
 sells a seashells at my the s


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python                              Rus
t
Hello world                              Hello world.                        Hel
lo, we're a heraldsess.
Good morning                             Good morning.                       Goo
d. More anything.
The quick brown fox                      The quick brown fox.                The
y...wicked brown fagus.
How are you doing today?                 How are you doing today?            How
 art you do wings to daddy?
She sells seashells by the seashore.     She sells seashells by the seasho   She
 sells a seashells at my the s
I read a book yesterday.                 I read a book yesterday.            I r
ead a book, yes, standard, say


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python                              Rus
t
Hello world                              Hello world.                        Hel
lo, we're a heraldsess.
Good morning                             Good morning.                       Goo
d. More anything.
The quick brown fox                      The quick brown fox.                The
y...wicked brown fagus.
How are you doing today?                 How are you doing today?            How
 art you do wings to daddy?
She sells seashells by the seashore.     She sells seashells by the seasho   She
 sells a seashells at my the s
I read a book yesterday.                 I read a book yesterday.            I r
ead a book, yes, standard, say
The dog is running in the park.          The dog is running in the park.     The
... the dog is running in the

Python: 7/7   Rust: 1/7


In [1]:
import subprocess, os, tempfile
import numpy as np
import soundfile as sf

REPO_ROOT = os.path.dirname(os.getcwd())
VOICERS_CLI = os.path.join(REPO_ROOT, "target/release/voicers-cli")
print(f"CLI exists: {os.path.exists(VOICERS_CLI)}")

import whisper

whisper_model = whisper.load_model("base")
print("Whisper: ready")

from mlx_audio.tts.utils import load as load_tts

py_model = load_tts("prince-canuma/Kokoro-82M")
print("Python Kokoro: ready")


def transcribe(wav_path):
    return whisper_model.transcribe(wav_path, language="en")["text"].strip()


def python_generate(text, voice="af_heart"):
    wav_path = tempfile.mktemp(suffix=".wav")
    for gen_result in py_model.generate(text, voice=voice, lang_code="a"):
        audio = np.array(gen_result.audio)
        sf.write(wav_path, audio, gen_result.sample_rate)
        return wav_path
    return None


def rust_generate(text, voice="af_heart"):
    wav_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [VOICERS_CLI, "--text", text, "--voice", voice, "--output", wav_path],
        capture_output=True,
        text=True,
        timeout=120,
    )
    phonemes = ""
    for line in r.stderr.splitlines():
        if line.strip().startswith("chunk"):
            phonemes += line.split(":", 1)[1].strip() + " "
    return wav_path, phonemes.strip()


print("All ready.")

CLI exists: True
Whisper: ready


Fetching 63 files:   0%|          | 0/63 [00:00<?, ?it/s]

Python Kokoro: ready
All ready.


In [2]:
# A/B Comparison: Python vs Rust (after iSTFT normalization fix)
test_phrases = [
    "Hello world",
    "Good morning",
    "The quick brown fox",
    "How are you doing today?",
    "She sells seashells by the seashore.",
    "I read a book yesterday.",
    "The dog is running in the park.",
]

print(f"{'Input':<40} {'Python Whisper':<40} {'Rust Whisper'}")
print("=" * 120)

py_results = []
rust_results = []

for phrase in test_phrases:
    py_wav = python_generate(phrase)
    py_heard = transcribe(py_wav) if py_wav else "FAILED"
    os.unlink(py_wav) if py_wav else None

    rust_wav, rust_ph = rust_generate(phrase)
    rust_heard = transcribe(rust_wav)
    os.unlink(rust_wav)

    py_results.append((phrase, py_heard))
    rust_results.append((phrase, rust_heard, rust_ph))

    print(f"{phrase:<40} {py_heard[:38]:<40} {rust_heard[:38]}")


# Score: exact match (case-insensitive, strip trailing punct)
def normalize(s):
    return s.lower().rstrip(".,!?;:")


py_score = sum(1 for p, h in py_results if normalize(p) == normalize(h))
rust_score = sum(1 for p, h, _ in rust_results if normalize(p) == normalize(h))
print(
    f"\nExact match: Python {py_score}/{len(test_phrases)}   Rust {rust_score}/{len(test_phrases)}"
)

Input                                    Python Whisper
  Rust Whisper
Creating new KokoroPipeline for language: a


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Input                                    Python Whisper
  Rust Whisper
Creating new KokoroPipeline for language: a
Hello world                              Hello world.
  Hello, we're told


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python Whisper
  Rust Whisper
Creating new KokoroPipeline for language: a
Hello world                              Hello world.
  Hello, we're told
Good morning                             Good morning.
  Good. More in singing.


...

Input                                    Python Whisper
  Rust Whisper
Creating new KokoroPipeline for language: a
Hello world                              Hello world.
  Hello, we're told
Good morning                             Good morning.
  Good. More in singing.
The quick brown fox                      The quick brown fox.
  The quick brown fogs.
How are you doing today?                 How are you doing today?
  How are you doing to hurt Zeta?
She sells seashells by the seashore.     She sells seashells by the seashore.
  She sells a sea she sells by a the sea


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python Whisper
  Rust Whisper
Creating new KokoroPipeline for language: a
Hello world                              Hello world.
  Hello, we're told
Good morning                             Good morning.
  Good. More in singing.
The quick brown fox                      The quick brown fox.
  The quick brown fogs.
How are you doing today?                 How are you doing today?
  How are you doing to hurt Zeta?
She sells seashells by the seashore.     She sells seashells by the seashore.
  She sells a sea she sells by a the sea
I read a book yesterday.                 I read a book yesterday.
  I read book. Yes, standard.


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python Whisper
  Rust Whisper
Creating new KokoroPipeline for language: a
Hello world                              Hello world.
  Hello, we're told
Good morning                             Good morning.
  Good. More in singing.
The quick brown fox                      The quick brown fox.
  The quick brown fogs.
How are you doing today?                 How are you doing today?
  How are you doing to hurt Zeta?
She sells seashells by the seashore.     She sells seashells by the seashore.
  She sells a sea she sells by a the sea
I read a book yesterday.                 I read a book yesterday.
  I read book. Yes, standard.
The dog is running in the park.          The dog is running in the park.
  Did it all go get... is this running i

Exact match: Python 7/7   Rust 0/7


In [3]:
# Compare the actual waveform statistics between Python and Rust
py_wav = python_generate("Hello world")
rust_wav, _ = rust_generate("Hello world")

py_audio, py_sr = sf.read(py_wav)
rust_audio, rust_sr = sf.read(rust_wav)

print(f"Python: shape={py_audio.shape}, sr={py_sr}, dtype={py_audio.dtype}")
print(
    f"  min={py_audio.min():.4f}, max={py_audio.max():.4f}, mean={py_audio.mean():.6f}, std={py_audio.std():.4f}"
)
print(f"  samples: {len(py_audio)}, duration: {len(py_audio) / py_sr:.2f}s")
print()
print(f"Rust:   shape={rust_audio.shape}, sr={rust_sr}, dtype={rust_audio.dtype}")
print(
    f"  min={rust_audio.min():.4f}, max={rust_audio.max():.4f}, mean={rust_audio.mean():.6f}, std={rust_audio.std():.4f}"
)
print(f"  samples: {len(rust_audio)}, duration: {len(rust_audio) / rust_sr:.2f}s")

# Amplitude ratio
print(f"\nAmplitude ratio (rust_std / py_std): {rust_audio.std() / py_audio.std():.4f}")
print(
    f"RMS ratio: {np.sqrt(np.mean(rust_audio**2)) / np.sqrt(np.mean(py_audio**2)):.4f}"
)

os.unlink(py_wav)
os.unlink(rust_wav)

Python: shape=(36600,), sr=24000, dtype=float64
  min=-0.2066, max=0.1968, mean=-0.000065, std=0.0340
  samples: 36600, duration: 1.52s

Rust:   shape=(60000,), sr=24000, dtype=float64
  min=-0.1710, max=0.2206, mean=-0.000004, std=0.0283
  samples: 60000, duration: 2.50s

Amplitude ratio (rust_std / py_std): 0.8318
RMS ratio: 0.8318


In [4]:
# Let's look at the WAV structure more carefully
# The Rust output being longer could mean:
# 1. Duration prediction is wrong (model predicting longer durations)
# 2. Extra silence at start/end (padding not trimmed)
# 3. Wrong sample rate in WAV header

# Check if there's a long silence tail in Rust output
py_wav = python_generate("Hello world")
rust_wav, _ = rust_generate("Hello world")
py_audio, _ = sf.read(py_wav)
rust_audio, _ = sf.read(rust_wav)


# Find where the audio actually starts/ends (above noise floor)
def find_audio_bounds(audio, threshold=0.005):
    abs_audio = np.abs(audio)
    above = abs_audio > threshold
    if not np.any(above):
        return 0, len(audio)
    start = np.argmax(above)
    end = len(audio) - np.argmax(above[::-1])
    return int(start), int(end)


py_start, py_end = find_audio_bounds(py_audio)
rust_start, rust_end = find_audio_bounds(rust_audio)

print(f"Python: total={len(py_audio)} samples")
print(
    f"  Audio region: {py_start} to {py_end} ({py_end - py_start} samples, {(py_end - py_start) / 24000:.2f}s)"
)
print(
    f"  Leading silence: {py_start / 24000:.3f}s, trailing silence: {(len(py_audio) - py_end) / 24000:.3f}s"
)
print()
print(f"Rust:   total={len(rust_audio)} samples")
print(
    f"  Audio region: {rust_start} to {rust_end} ({rust_end - rust_start} samples, {(rust_end - rust_start) / 24000:.2f}s)"
)
print(
    f"  Leading silence: {rust_start / 24000:.3f}s, trailing silence: {(len(rust_audio) - rust_end) / 24000:.3f}s"
)

os.unlink(py_wav)
os.unlink(rust_wav)

Python: total=36600 samples
  Audio region: 9245 to 25779 (16534 samples, 0.69s)
  Leading silence: 0.385s, trailing silence: 0.451s

Rust:   total=60600 samples
  Audio region: 11183 to 52186 (41003 samples, 1.71s)
  Leading silence: 0.466s, trailing silence: 0.351s


In [5]:
# Let's trace the exact intermediate values.
# First, check what phonemes and token IDs each side uses for "Hello world"

# Python side: get phonemes and token IDs
script = """
import sys
sys.path.insert(0, '/Users/kylekelley/conductor/workspaces/misaki/abuja-v4')
from misaki.en import G2P
import spacy, json

g2p = G2P(trf=False, british=False, unk='')
ps, tokens = g2p("Hello world")
print(f"phonemes: {ps}")
print(f"tokens: {[(t.text, t.phonemes, t.tag) for t in tokens]}")

# Show the vocab mapping
import json as j
from huggingface_hub import hf_hub_download
config_path = hf_hub_download('prince-canuma/Kokoro-82M', 'config.json')
with open(config_path) as f:
    config = j.load(f)
vocab = config['vocab']
ids = [vocab.get(c, -1) for c in ps]
print(f"token_ids: {ids}")
print(f"phoneme_count: {len(ps)}")
"""
r = subprocess.run(
    [
        "uv",
        "run",
        "--with",
        "misaki[en]",
        "--with",
        "spacy",
        "--with",
        "transformers",
        "--with",
        "torch",
        "--with",
        "en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl",
        "--with",
        "huggingface-hub",
        "python",
        "-c",
        script,
    ],
    capture_output=True,
    text=True,
    timeout=120,
)
print("=== Python misaki output ===")
print(r.stdout)
if r.stderr:
    # Just show the important lines
    for line in r.stderr.splitlines():
        if not line.startswith((" ", "  ")) and "WARNING" not in line:
            print(f"  stderr: {line}")

=== Python misaki output ===
phonemes: həlˈO wˈɜɹld
tokens: [('Hello', 'həlˈO', 'UH'), ('world', 'wˈɜɹld', 'NN')]
token_ids: [50, 83, 54, 156, 31, 16, 65, 156, 87, 123, 54, 46]
phoneme_count: 12

  stderr:
  stderr: Loading weights:   0%|                                  | 0/50 [00:00<
?, ?it/s]
  stderr: Loading weights: 100%|██████████████████████| 50/50 [00:00<00:00, 1156
2.84it/s]


In [6]:
# Now check what the Rust side produces for "Hello world"
# The phonemes were: həlˈO wˈɜɹld

# Let's check token IDs from Rust by looking at the vocab
import json
from huggingface_hub import hf_hub_download

config_path = hf_hub_download("prince-canuma/Kokoro-82M", "config.json")
with open(config_path) as f:
    config = json.load(f)
vocab = config["vocab"]

rust_phonemes = "həlˈO wˈɜɹld"
rust_ids = [vocab.get(c, -1) for c in rust_phonemes]
print(f"Rust phonemes: {rust_phonemes}")
print(f"Rust token_ids: {rust_ids}")
print(f"Rust phoneme_count: {len(rust_phonemes)}")

py_phonemes = "həlˈO wˈɜɹld"
py_ids = [vocab.get(c, -1) for c in py_phonemes]
print(f"\nPython phonemes: {py_phonemes}")
print(f"Python token_ids: {py_ids}")
print(f"Python phoneme_count: {len(py_phonemes)}")

print(f"\nPhonemes match: {rust_phonemes == py_phonemes}")
print(f"Token IDs match: {rust_ids == py_ids}")

# Check vocab for space
print(f"\nSpace in vocab: {vocab.get(' ', 'NOT FOUND')}")
print(f"BOS token (0): {[k for k, v in vocab.items() if v == 0]}")

# Show full vocab
print(f"\nVocab size: {len(vocab)}")
print(f"Vocab: {dict(sorted(vocab.items(), key=lambda x: x[1]))}")

Rust phonemes: həlˈO wˈɜɹld
Rust token_ids: [50, 83, 54, 156, 31, 16, 65, 156, 87, 123, 54, 46]
Rust phoneme_count: 12

Python phonemes: həlˈO wˈɜɹld
Python token_ids: [50, 83, 54, 156, 31, 16, 65, 156, 87, 123, 54, 46]
Python phoneme_count: 12

Phonemes match: True
Token IDs match: True

Space in vocab: 16
BOS token (0): []

Vocab size: 114
Vocab: {';': 1, ':': 2, ',': 3, '.': 4, '!': 5, '?': 6, '—': 9, '…': 10, '"': 11
, '(': 12, ')': 13, '“': 14, '”': 15, ' ': 16, '̃': 17, 'ʣ': 18, 'ʥ': 19, 'ʦ': 20
, 'ʨ': 21, 'ᵝ': 22, 'ꭧ': 23, 'A': 24, 'I': 25, 'O': 31, 'Q': 33, 'S': 35, 'T': 3
6, 'W': 39, 'Y': 41, 'ᵊ': 42, 'a': 43, 'b': 44, 'c': 45, 'd': 46, 'e': 47, 'f':
48, 'h': 50, 'i': 51, 'j': 52, 'k': 53, 'l': 54, 'm': 55, 'n': 56, 'o': 57, 'p':
 58, 'q': 59, 'r': 60, 's': 61, 't': 62, 'u': 63, 'v': 64, 'w': 65, 'x': 66, 'y'
: 67, 'z': 68, 'ɑ': 69, 'ɐ': 70, 'ɒ': 71, 'æ': 72, 'β': 75, 'ɔ': 76, 'ɕ': 77, 'ç
': 78, 'ɖ': 80, 'ð': 81, 'ʤ': 82, 'ə': 83, 'ɚ': 85, 'ɛ': 86, 'ɜ': 87, 'ɟ': 90, '
ɡ': 92, 

In [7]:
# Deep dive: extract intermediate tensors from both pipelines
# Python: Use the model directly to get duration predictions

script = """
import sys, json, numpy as np
sys.path.insert(0, '/Users/kylekelley/conductor/workspaces/misaki/abuja-v4')
sys.path.insert(0, '/Users/kylekelley/conductor/workspaces/kokoro/montevideo')

import torch
from kokoro.model import KModel
from huggingface_hub import hf_hub_download

# Load model
config_path = hf_hub_download('prince-canuma/Kokoro-82M', 'config.json')
with open(config_path) as f:
    config = json.load(f)

model = KModel(config)
weights_path = hf_hub_download('prince-canuma/Kokoro-82M', 'kokoro-v1_0.pth')
state = torch.load(weights_path, map_location='cpu', weights_only=True)
model.load_state_dict(state)
model.eval()

# Load voice
voice_path = hf_hub_download('prince-canuma/Kokoro-82M', 'voices/af_heart.pt')
voice = torch.load(voice_path, map_location='cpu', weights_only=True)

# Prepare input - same phonemes as Rust
phonemes = "həlˈO wˈɜɹld"
vocab = config['vocab']
tokens = [vocab[c] for c in phonemes]
input_ids = torch.tensor([[0, *tokens, 0]], dtype=torch.long)
ref_s = voice[len(tokens)]  # voice_pack indexed by phoneme count

print(f"input_ids shape: {input_ids.shape}")
print(f"input_ids: {input_ids.tolist()}")
print(f"ref_s shape: {ref_s.shape}")
print(f"ref_s[:5]: {ref_s[0,:5].tolist()}")

# Run the model forward pass with torch.no_grad
with torch.no_grad():
    # Get BERT output
    bert_out = model.bert(input_ids, attention_mask=(~(input_ids==0)).int())
    asr = model.bert_encoder(bert_out[0]).transpose(1, 2)
    print(f"asr shape: {asr.shape}")
    print(f"asr mean: {asr.mean().item():.6f}, std: {asr.std().item():.6f}")
    
    # Split style
    s_content = ref_s[:, 128:]
    s_ref = ref_s[:, :128]
    
    # Get durations via predictor
    d = model.predictor.text_encoder(asr, s_content)
    print(f"duration_encoded shape: {d.shape}")
    
    # Full forward pass to get audio
    audio = model(input_ids, ref_s, speed=1.0)
    if isinstance(audio, tuple):
        audio = audio[0]
    audio_np = audio.cpu().numpy().flatten()
    print(f"audio shape: {audio_np.shape}, duration: {len(audio_np)/24000:.2f}s")
    print(f"audio stats: min={audio_np.min():.4f}, max={audio_np.max():.4f}, std={audio_np.std():.4f}")
"""

r = subprocess.run(
    [
        "uv",
        "run",
        "--with",
        "misaki[en]",
        "--with",
        "spacy",
        "--with",
        "transformers",
        "--with",
        "torch",
        "--with",
        "huggingface-hub",
        "--with",
        "soundfile",
        "--with",
        "en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl",
        "python",
        "-c",
        script,
    ],
    capture_output=True,
    text=True,
    timeout=120,
)
print("=== Python PyTorch model internals ===")
for line in r.stdout.splitlines():
    print(line)
if r.returncode != 0:
    print("ERRORS:")
    for line in r.stderr.splitlines()[-10:]:
        print(f"  {line}")

=== Python PyTorch model internals ===
ERRORS:
  Traceback (most recent call last):
    File "<string>", line 7, in <module>
    File "/Users/kylekelley/conductor/workspaces/kokoro/montevideo/kokoro/__init
__.py", line 3, in <module>
      from loguru import logger
  ModuleNotFoundError: No module named 'loguru'


In [8]:
# Add loguru + phonemizer deps
script = """
import sys, json, numpy as np
sys.path.insert(0, '/Users/kylekelley/conductor/workspaces/kokoro/montevideo')

import torch
from kokoro.model import KModel
from huggingface_hub import hf_hub_download

config_path = hf_hub_download('prince-canuma/Kokoro-82M', 'config.json')
with open(config_path) as f:
    config = json.load(f)

model = KModel(config)
weights_path = hf_hub_download('prince-canuma/Kokoro-82M', 'kokoro-v1_0.pth')
state = torch.load(weights_path, map_location='cpu', weights_only=True)
model.load_state_dict(state)
model.eval()

voice_path = hf_hub_download('prince-canuma/Kokoro-82M', 'voices/af_heart.pt')
voice = torch.load(voice_path, map_location='cpu', weights_only=True)

phonemes = "həlˈO wˈɜɹld"
vocab = config['vocab']
tokens = [vocab[c] for c in phonemes]
input_ids = torch.tensor([[0, *tokens, 0]], dtype=torch.long)
ref_s = voice[len(tokens)]

print(f"input_ids shape: {input_ids.shape}")
print(f"input_ids: {input_ids.tolist()}")
print(f"ref_s shape: {ref_s.shape}")
print(f"ref_s[0,:5]: {ref_s[0,:5].tolist()}")

with torch.no_grad():
    audio = model(input_ids, ref_s, speed=1.0)
    if isinstance(audio, tuple):
        audio = audio[0]
    audio_np = audio.cpu().numpy().flatten()
    print(f"audio shape: {audio_np.shape}, duration: {len(audio_np)/24000:.2f}s")
    print(f"audio stats: min={audio_np.min():.4f}, max={audio_np.max():.4f}, std={audio_np.std():.4f}")
"""

r = subprocess.run(
    [
        "uv",
        "run",
        "--with",
        "loguru",
        "--with",
        "phonemizer",
        "--with",
        "transformers",
        "--with",
        "torch",
        "--with",
        "huggingface-hub",
        "--with",
        "soundfile",
        "--with",
        "misaki[en]",
        "--with",
        "spacy",
        "--with",
        "en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl",
        "python",
        "-c",
        script,
    ],
    capture_output=True,
    text=True,
    timeout=120,
)
print("=== PyTorch Kokoro model ===")
for line in r.stdout.splitlines():
    print(line)
if r.returncode != 0:
    print("ERRORS:")
    for line in r.stderr.splitlines()[-10:]:
        print(f"  {line}")

=== PyTorch Kokoro model ===
ERRORS:
  Traceback (most recent call last):
    File "<string>", line 13, in <module>
    File "/Users/kylekelley/conductor/workspaces/kokoro/montevideo/kokoro/model.
py", line 46, in __init__
      config = hf_hub_download(repo_id=repo_id, filename='config.json')
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "/Users/kylekelley/.cache/uv/archive-v0/zEHrRvVr0g5qocUwDDoKJ/lib/pytho
n3.12/site-packages/huggingface_hub/utils/_validators.py", line 85, in _inner_fn
      validate_repo_id(arg_value)
    File "/Users/kylekelley/.cache/uv/archive-v0/zEHrRvVr0g5qocUwDDoKJ/lib/pytho
n3.12/site-packages/huggingface_hub/utils/_validators.py", line 130, in validate
_repo_id
      raise HFValidationError(f"Repo id must be a string, not {type(repo_id)}: '
{repo_id}'.")
  huggingface_hub.errors.HFValidationError: Repo id must be a string, not <class
 'dict'>: '{'istftnet': {'upsample_kernel_sizes': [20, 12], 'upsample_rates': [1
0, 6], 'g

In [9]:
# Use the mlx-audio Python model we already have loaded to get intermediate info
# The py_model is a loaded KokoroPipeline

# Let's check the Python mlx-audio model internals
import mlx.core as mx

# Get the actual model from the pipeline
pipeline = py_model
model = pipeline.model  # The underlying MLX model
print(f"Model type: {type(model)}")
print(f"Model attributes: {[a for a in dir(model) if not a.startswith('_')]}")

# Check if we can access the forward pass components
vocab = model.vocab if hasattr(model, "vocab") else None
print(f"Has vocab: {vocab is not None}")

# Try to trace the model's generate method
phonemes = "həlˈO wˈɜɹld"
if vocab:
    input_ids = [vocab.get(c, -1) for c in phonemes]
    print(f"input_ids: {input_ids}")
    print(f"phoneme count: {len(input_ids)}")

AttributeError: 'Model' object has no attribute 'model'

In [10]:
# py_model is the Model itself, not a pipeline wrapper
import mlx.core as mx

model = py_model
print(f"Model type: {type(model)}")
print(f"Model attributes: {[a for a in dir(model) if not a.startswith('_')][:30]}")

# Look at the generate method source
import inspect

# Get the __call__ or generate method
print(f"\nHas __call__: {hasattr(model, '__call__')}")
print(f"Has generate: {hasattr(model, 'generate')}")

# Check key attributes
for attr in ["vocab", "bert", "bert_encoder", "predictor", "text_encoder", "decoder"]:
    print(f"  {attr}: {hasattr(model, attr)}")

Model type: <class 'mlx_audio.tts.models.kokoro.kokoro.Model'>
Model attributes: ['Output', 'REPO_ID', 'apply', 'apply_to_modules', 'children',
 'clear', 'config', 'context_length', 'copy', 'eval', 'filter_and_map', 'freeze'
, 'fromkeys', 'generate', 'get', 'is_module', 'items', 'keys', 'leaf_modules', '
load_weights', 'modules', 'named_modules', 'parameters', 'pop', 'popitem', 'repo
_id', 'sample_rate', 'sanitize', 'save_weights', 'set_dtype']

Has __call__: True
Has generate: True
  vocab: True
  bert: True
  bert_encoder: True
  predictor: True
  text_encoder: True
  decoder: True


In [11]:
# Now let's trace the forward pass step by step in Python MLX
# First, read the Python model's __call__ to understand the flow
import inspect

src = inspect.getsource(model.__call__)
print(src[:3000])

    def __call__(
        self,
        phonemes: str,
        ref_s: mx.array,
        speed: Number = 1,
        return_output: bool = False,  # MARK: BACKWARD COMPAT
        decoder=None,
    ) -> Union["KokoroModel.Output", mx.array]:
        input_ids = list(
            filter(lambda i: i is not None, map(lambda p: self.vocab.get(p), pho
nemes))
        )
        assert len(input_ids) + 2 <= self.context_length, (
            len(input_ids) + 2,
            self.context_length,
        )
        input_ids = mx.array([[0, *input_ids, 0]])
        input_lengths = mx.array([input_ids.shape[-1]])
        text_mask = mx.arange(int(input_lengths.max()))[None, ...]
        text_mask = mx.repeat(text_mask, input_lengths.shape[0], axis=0).astype(
            input_lengths.dtype
        )
        text_mask = text_mask + 1 > input_lengths[:, None]
        bert_dur, _ = self.bert(input_ids, attention_mask=(~text_mask).astype(mx
.int32))
        d_en = self.bert_encoder(bert_dur).transpose(0,

In [12]:
# Step-by-step instrumented forward pass in Python MLX
import mlx.core as mx
import numpy as np

phonemes = "həlˈO wˈɜɹld"
vocab = model.vocab
input_ids_list = [v for c in phonemes if (v := vocab.get(c)) is not None]
input_ids = mx.array([[0, *input_ids_list, 0]])
seq_len = input_ids.shape[-1]
input_lengths = mx.array([seq_len])

text_mask = mx.arange(int(input_lengths.max()))[None, :]
text_mask = mx.repeat(text_mask, input_lengths.shape[0], axis=0).astype(
    input_lengths.dtype
)
text_mask = text_mask + 1 > input_lengths[:, None]

print(f"input_ids: {input_ids.tolist()}")
print(f"seq_len: {seq_len}")
print(f"text_mask: {text_mask.tolist()}")
print(f"text_mask shape: {text_mask.shape}")

# Load voice
from huggingface_hub import hf_hub_download

voice_path = hf_hub_download("prince-canuma/Kokoro-82M", "voices/af_heart.pt")
import torch

voice_pt = torch.load(voice_path, map_location="cpu", weights_only=True)
voice = mx.array(voice_pt.numpy())
ref_s = voice[len(input_ids_list)]
print(f"\nvoice pack shape: {voice.shape}")
print(f"ref_s shape (indexed by phoneme count {len(input_ids_list)}): {ref_s.shape}")
print(f"ref_s[0,:5]: {ref_s[0, :5].tolist()}")

input_ids: [[0, 50, 83, 54, 156, 31, 16, 65, 156, 87, 123, 54, 46, 0]]
seq_len: 14
text_mask: [[False, False, False, False, False, False, False, False, False, Fals
e, False, False, False, False]]
text_mask shape: (1, 14)

voice pack shape: (510, 1, 256)
ref_s shape (indexed by phoneme count 12): (1, 256)
ref_s[0,:5]: [-0.23354966938495636, 0.19597655534744263, -0.003190262010321021,
-0.12491673231124878, -0.23539023101329803]


In [13]:
# BERT + duration prediction
attention_mask = (~text_mask).astype(mx.int32)
bert_dur, _ = model.bert(input_ids, attention_mask=attention_mask)
d_en = model.bert_encoder(bert_dur).transpose(0, 2, 1)

s = ref_s[:, 128:]  # prosody style

print(f"bert_dur shape: {bert_dur.shape}")
print(f"d_en shape: {d_en.shape}")
print(f"s shape: {s.shape}")
print(
    f"d_en mean: {float(mx.mean(d_en)):.6f}, std: {float(mx.sqrt(mx.mean(d_en * d_en))):.6f}"
)

# Duration encoding
d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)
print(f"\nd (duration encoded) shape: {d.shape}")
print(f"d mean: {float(mx.mean(d)):.6f}")

# Duration LSTM
x, _ = model.predictor.lstm(d)
print(f"lstm out shape: {x.shape}")

# Duration projection
duration = model.predictor.duration_proj(x)
duration = mx.sigmoid(duration).sum(axis=-1) / 1.0  # speed=1.0
pred_dur = mx.clip(mx.round(duration), a_min=1, a_max=None).astype(mx.int32)[0]
mx.eval(pred_dur)

print(f"\nduration (pre-round): {duration[0].tolist()}")
print(f"pred_dur: {pred_dur.tolist()}")
print(f"total_frames: {int(mx.sum(pred_dur))}")

bert_dur shape: (1, 14, 768)
d_en shape: (1, 512, 14)
s shape: (1, 128)
d_en mean: -0.016284, std: 1.045752

d (duration encoded) shape: (1, 14, 640)
d mean: -0.006935
lstm out shape: (1, 14, 512)

duration (pre-round): [17.028810501098633, 2.013627529144287, 2.0196692943573, 1
.9970686435699463, 2.0101888179779053, 2.073147773742676, 2.0227818489074707, 1.
422141671180725, 2.0004937648773193, 2.462941884994507, 4.1396098136901855, 3.19
56851482391357, 11.702914237976074, 7.928163528442383]
pred_dur: [17, 2, 2, 2, 2, 2, 2, 1, 2, 2, 4, 3, 12, 8]
total_frames: 61


In [14]:
# Get Rust intermediate values - add a debug flag that prints durations
# Let's modify the CLI temporarily to dump duration info

# Actually, easier: write a tiny Rust test that prints the intermediate values
# For now, let's just check the Rust output duration to compare

rust_wav, rust_ph = rust_generate("Hello world")
rust_audio, _ = sf.read(rust_wav)

# From Python: total_frames=61, with hop_size=300 (upsample 10*6*5=300)
# expected samples = 61 * 300 = 18300 (plus padding)
# Python produces 36600 samples
# Let's check what the Rust side looks like

# Actually the istft hop is gen_istft_hop_size=5, and upsample is 10*6=60
# So effective hop = 60 * 5 = 300 samples per frame
# 61 frames * 300 = 18300

print(f"Python: 61 frames * 300 hop = 18300 expected samples")
print(f"Python actual: 36600 samples (2x expected - likely due to center padding)")
print(f"Rust actual: {len(rust_audio)} samples")
print(f"Rust / Python ratio: {len(rust_audio) / 36600:.2f}")

# If Rust is 60000 samples and Python is 36600, and Python has 61 frames...
# Rust would have about 61 * (60000/36600) = ~100 frames
# That suggests durations are ~1.64x too long
print(f"\nIf proportional, Rust frames ~ {61 * len(rust_audio) / 36600:.0f}")
os.unlink(rust_wav)

Python: 61 frames * 300 hop = 18300 expected samples
Python actual: 36600 samples (2x expected - likely due to center padding)
Rust actual: 58800 samples
Rust / Python ratio: 1.61

If proportional, Rust frames ~ 98


In [15]:
# Compare intermediate values between Python and Rust

print("=== d_en (BERT encoder output) ===")
print(f"  Python: shape (1,512,14), mean: -0.016284, rms: 1.045752")
print(f"  Rust:   shape [1,512,14], mean:  0.000983, rms: 1.073082")
print(f"  -> Mean is different sign! RMS similar. BERT output diverges slightly.")

print("\n=== ref_s (voice embedding) ===")
print(f"  Python first5: [-0.2335, 0.1960, -0.0032, -0.1249, -0.2354]")
print(f"  Rust   first5: [-0.2342, 0.1981, -0.0026, -0.1251, -0.2329]")
print(f"  -> Very close! Voice loading is correct.")

print("\n=== s (prosody style, ref_s[:, 128:]) ===")
print(f"  Python: (checked via ref_s[0,128:133])")

# Actually let me get the Python s values
s_vals = ref_s[0, 128:133].tolist()
print(f"  Python first5: {s_vals}")
print(f"  Rust   first5: [-0.0819, 0.3838, 0.8232, 0.2056, -0.0507]")
print(f"  -> Very close! Style extraction correct.")

print("\n=== d (duration encoder output) ===")
print(f"  Python: shape (1,14,640), mean: -0.006935")
print(f"  Rust:   shape [1,14,640], mean: -0.002426")
print(f"  -> Starting to diverge (different d_en inputs -> different d)")

print("\n=== duration (pre-round) ===")
py_dur = [
    17.03,
    2.01,
    2.02,
    2.00,
    2.01,
    2.07,
    2.02,
    1.42,
    2.00,
    2.46,
    4.14,
    3.20,
    11.70,
    7.93,
]
rust_dur = [
    17.66,
    3.48,
    3.06,
    2.27,
    3.25,
    8.04,
    17.69,
    2.30,
    3.39,
    14.92,
    3.43,
    2.17,
    6.06,
    13.86,
]

print(f"  Python: {py_dur}")
print(f"  Rust:   {rust_dur}")
print(f"  Ratios: {[f'{r / p:.2f}' for p, r in zip(py_dur, rust_dur)]}")
print(f"  -> WILDLY different! Esp: space(2->18), ɜ(2.5->15), d(12->6), EOS(8->14)")
print(
    f"  -> The BERT d_en difference cascades through duration encoder -> LSTM -> projection"
)

print("\n=== pred_dur ===")
py_pred = [17, 2, 2, 2, 2, 2, 2, 1, 2, 2, 4, 3, 12, 8]
rust_pred = [18, 3, 3, 2, 3, 8, 18, 2, 3, 15, 3, 2, 6, 14]
print(f"  Python: {py_pred} -> total {sum(py_pred)}")
print(f"  Rust:   {rust_pred} -> total {sum(rust_pred)}")

print("\n=== ROOT CAUSE ===")
print("The BERT (ALBERT) encoder produces different output in Rust vs Python.")
print(
    "Small differences in d_en get amplified by the duration encoder + LSTM pipeline."
)
print("Need to investigate the CustomAlbert implementation.")

=== d_en (BERT encoder output) ===
  Python: shape (1,512,14), mean: -0.016284, rms: 1.045752
  Rust:   shape [1,512,14], mean:  0.000983, rms: 1.073082
  -> Mean is different sign! RMS similar. BERT output diverges slightly.

=== ref_s (voice embedding) ===
  Python first5: [-0.2335, 0.1960, -0.0032, -0.1249, -0.2354]
  Rust   first5: [-0.2342, 0.1981, -0.0026, -0.1251, -0.2329]
  -> Very close! Voice loading is correct.

=== s (prosody style, ref_s[:, 128:]) ===
  Python: (checked via ref_s[0,128:133])
  Python first5: [-0.07459218055009842, 0.3677627742290497, 0.7973013520240784,
0.20154288411140442, -0.05479338392615318]
  Rust   first5: [-0.0819, 0.3838, 0.8232, 0.2056, -0.0507]
  -> Very close! Style extraction correct.

=== d (duration encoder output) ===
  Python: shape (1,14,640), mean: -0.006935
  Rust:   shape [1,14,640], mean: -0.002426
  -> Starting to diverge (different d_en inputs -> different d)

=== duration (pre-round) ===
  Python: [17.03, 2.01, 2.02, 2.0, 2.01, 2.07

In [16]:
# Check weight keys for the duration encoder
from safetensors.numpy import load_file
from huggingface_hub import hf_hub_download

path = hf_hub_download("prince-canuma/Kokoro-82M", "model.safetensors")
weights = load_file(path)
print("=== predictor.text_encoder weights ===")
for k in sorted(weights.keys()):
    if "predictor.text_encoder" in k:
        print(f"  {k}: {weights[k].shape}")

print("\n=== predictor.lstm weights ===")
for k in sorted(weights.keys()):
    if k.startswith("predictor.lstm"):
        print(f"  {k}: {weights[k].shape}")

RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-69ba1dfe-55c9f7b132c9d7d8635e5dd6;740c0d2e-3e56-4985-9ddd-844fd55c68c4)

Entry Not Found for url: https://huggingface.co/prince-canuma/Kokoro-82M/resolve/main/model.safetensors.

In [17]:
# Check weight keys using the local cache
from safetensors.numpy import load_file
import os

SNAPSHOT = os.path.expanduser(
    "~/.cache/huggingface/hub/models--prince-canuma--Kokoro-82M/snapshots/e02c9eada7ce7416798af36b190a8a2dd2ecd566"
)
weights = load_file(os.path.join(SNAPSHOT, "kokoro-v1_0.safetensors"))

print("=== predictor.text_encoder weights ===")
for k in sorted(weights.keys()):
    if "predictor.text_encoder" in k:
        print(f"  {k}: {weights[k].shape}")

print("\n=== predictor.lstm weights ===")
for k in sorted(weights.keys()):
    if k.startswith("predictor.lstm"):
        print(f"  {k}: {weights[k].shape}")

print("\n=== predictor.duration_proj ===")
for k in sorted(weights.keys()):
    if "duration_proj" in k:
        print(f"  {k}: {weights[k].shape}")

=== predictor.text_encoder weights ===
  predictor.text_encoder.lstms.0.bias_hh_l0: (1024,)
  predictor.text_encoder.lstms.0.bias_hh_l0_reverse: (1024,)
  predictor.text_encoder.lstms.0.bias_ih_l0: (1024,)
  predictor.text_encoder.lstms.0.bias_ih_l0_reverse: (1024,)
  predictor.text_encoder.lstms.0.weight_hh_l0: (1024, 256)
  predictor.text_encoder.lstms.0.weight_hh_l0_reverse: (1024, 256)
  predictor.text_encoder.lstms.0.weight_ih_l0: (1024, 640)
  predictor.text_encoder.lstms.0.weight_ih_l0_reverse: (1024, 640)
  predictor.text_encoder.lstms.1.fc.bias: (1024,)
  predictor.text_encoder.lstms.1.fc.weight: (1024, 128)
  predictor.text_encoder.lstms.2.bias_hh_l0: (1024,)
  predictor.text_encoder.lstms.2.bias_hh_l0_reverse: (1024,)
  predictor.text_encoder.lstms.2.bias_ih_l0: (1024,)
  predictor.text_encoder.lstms.2.bias_ih_l0_reverse: (1024,)
  predictor.text_encoder.lstms.2.weight_hh_l0: (1024, 256)
  predictor.text_encoder.lstms.2.weight_hh_l0_reverse: (1024, 256)
  predictor.text_enco

In [18]:
# Instrument Python DurationEncoder step by step
import mlx.core as mx

# Re-use the already-computed values from earlier cells
# d_en, s, input_lengths, text_mask are all still in scope

# Manually run DurationEncoder
dur_enc = model.predictor.text_encoder

x = d_en  # (1, 512, 14)
style = s  # (1, 128)
m = text_mask  # (1, 14)

x = x.transpose(2, 0, 1)  # (14, 1, 512)
s_broad = mx.broadcast_to(style, (x.shape[0], x.shape[1], style.shape[-1]))
x = mx.concatenate([x, s_broad], axis=-1)  # (14, 1, 640)
x = mx.where(m[..., None].transpose(1, 0, 2), 0.0, x)
x = x.transpose(1, 2, 0)  # (1, 640, 14)

print(f"Initial: shape={x.shape}, mean={float(mx.mean(x)):.6f}")

for bi, block in enumerate(dur_enc.lstms):
    mx.eval(x)
    print(f"  Block {bi} input: shape={x.shape}, mean={float(mx.mean(x)):.6f}")

    if hasattr(block, "fc"):  # AdaLayerNorm
        x = block(x.transpose(0, 2, 1), style).transpose(0, 2, 1)
        x = mx.concatenate([x, s_broad.transpose(1, 2, 0)], axis=1)
        x = mx.where(m[..., None].transpose(0, 2, 1), 0.0, x)
    else:  # LSTM
        x_for_lstm = x.transpose(0, 2, 1)[0]  # (14, 640)
        x_lstm_out, _ = block(x_for_lstm)
        x = x_lstm_out.transpose(0, 2, 1)
        x_pad = mx.zeros([x.shape[0], x.shape[1], m.shape[-1]])
        x_pad[:, :, : x.shape[-1]] = x
        x = x_pad

mx.eval(x)
x_final = x.transpose(0, 2, 1)
print(f"\nFinal: shape={x_final.shape}, mean={float(mx.mean(x_final)):.6f}")

Initial: shape=(1, 640, 14), mean=-0.005801
  Block 0 input: shape=(1, 640, 14), mean=-0.005801
  Block 1 input: shape=(1, 512, 14), mean=-0.005467
  Block 2 input: shape=(1, 640, 14), mean=0.024576
  Block 3 input: shape=(1, 512, 14), mean=0.001655
  Block 4 input: shape=(1, 640, 14), mean=-0.001214
  Block 5 input: shape=(1, 512, 14), mean=-0.007109

Final: shape=(1, 14, 640), mean=-0.006935


In [19]:
# Check the actual weight values for the first DurationEncoder LSTM
lstm0 = dur_enc.lstms[0]
print(f"Type: {type(lstm0)}")
print(f"Wx_forward shape: {lstm0.Wx_forward.shape}")
print(f"Wh_forward shape: {lstm0.Wh_forward.shape}")
print(
    f"bias_ih_forward: {lstm0.bias_ih_forward.shape if lstm0.bias_ih_forward is not None else None}"
)
print(
    f"bias_hh_forward: {lstm0.bias_hh_forward.shape if lstm0.bias_hh_forward is not None else None}"
)

# Get first few values of Wx_forward
wx_fwd = lstm0.Wx_forward
mx.eval(wx_fwd)
print(f"\nWx_forward[0,:5]: {wx_fwd[0, :5].tolist()}")

# Combined bias
bias_combined = lstm0.bias_ih_forward + lstm0.bias_hh_forward
mx.eval(bias_combined)
print(f"Combined bias[:5]: {bias_combined[:5].tolist()}")

Type: <class 'mlx_audio.tts.models.kokoro.modules.LSTM'>
Wx_forward shape: (1024, 640)
Wh_forward shape: (1024, 256)
bias_ih_forward: (1024,)
bias_hh_forward: (1024,)

Wx_forward[0,:5]: [0.0005577385309152305, -0.041339144110679626, 0.1754166036844
2535, -0.10248763114213943, 0.06687430292367935]
Combined bias[:5]: [0.004875728860497475, -0.17237330973148346, 0.09251148998737
335, -0.01823487877845764, 0.2256598174571991]


In [20]:
# Trace exact shapes through the Python DurationEncoder
x = d_en  # (1, 512, 14)
style_in = s
m = text_mask

x = x.transpose(2, 0, 1)  # (14, 1, 512)
s_broad = mx.broadcast_to(style_in, (x.shape[0], x.shape[1], style_in.shape[-1]))
x = mx.concatenate([x, s_broad], axis=-1)  # (14, 1, 640)
x = mx.where(m[..., None].transpose(1, 0, 2), 0.0, x)
x = x.transpose(1, 2, 0)  # (1, 640, 14)
print(f"After init: {x.shape}")

for bi, block in enumerate(dur_enc.lstms):
    if hasattr(block, "fc"):  # AdaLayerNorm
        xt = x.transpose(0, 2, 1)
        print(f"  Block {bi} (Norm): input shape {xt.shape}")
        x = block(xt, style_in).transpose(0, 2, 1)
        print(f"    after norm: {x.shape}")
        x = mx.concatenate([x, s_broad.transpose(1, 2, 0)], axis=1)
        print(f"    after concat: {x.shape}")
        x = mx.where(m[..., None].transpose(0, 2, 1), 0.0, x)
    else:  # LSTM
        xt = x.transpose(0, 2, 1)
        print(f"  Block {bi} (LSTM): pre-transpose shape {xt.shape}")
        xt0 = xt[0]
        print(f"    after [0]: {xt0.shape}")
        h, _ = block(xt0)
        print(f"    LSTM output: {h.shape}")
        x = h.transpose(0, 2, 1) if h.ndim == 3 else h.T
        print(f"    after transpose: {x.shape}")
        x_pad = mx.zeros([x.shape[0], x.shape[1], m.shape[-1]])
        x_pad = x_pad.at[:, :, : x.shape[-1]].add(x)
        x = x_pad
        print(f"    after pad: {x.shape}")

print(f"\nFinal: {x.transpose(0, 2, 1).shape}")

After init: (1, 640, 14)
  Block 0 (LSTM): pre-transpose shape (1, 14, 640)
    after [0]: (14, 640)
    LSTM output: (1, 14, 512)
    after transpose: (1, 512, 14)
    after pad: (1, 512, 14)
  Block 1 (Norm): input shape (1, 14, 512)
    after norm: (1, 512, 14)
    after concat: (1, 640, 14)
  Block 2 (LSTM): pre-transpose shape (1, 14, 640)
    after [0]: (14, 640)
    LSTM output: (1, 14, 512)
    after transpose: (1, 512, 14)
    after pad: (1, 512, 14)
  Block 3 (Norm): input shape (1, 14, 512)
    after norm: (1, 512, 14)
    after concat: (1, 640, 14)
  Block 4 (LSTM): pre-transpose shape (1, 14, 640)
    after [0]: (14, 640)
    LSTM output: (1, 14, 512)
    after transpose: (1, 512, 14)
    after pad: (1, 512, 14)
  Block 5 (Norm): input shape (1, 14, 512)
    after norm: (1, 512, 14)
    after concat: (1, 640, 14)

Final: (1, 14, 640)


In [21]:
# Check backward weights
lstm0 = dur_enc.lstms[0]
wx_bwd = lstm0.Wx_backward
mx.eval(wx_bwd)
print(f"Wx_backward[0,:5]: {wx_bwd[0, :5].tolist()}")

bias_bwd = lstm0.bias_ih_backward + lstm0.bias_hh_backward
mx.eval(bias_bwd)
print(f"Combined backward bias[:5]: {bias_bwd[:5].tolist()}")

wh_fwd = lstm0.Wh_forward
mx.eval(wh_fwd)
print(f"Wh_forward[0,:5]: {wh_fwd[0, :5].tolist()}")

wh_bwd = lstm0.Wh_backward
mx.eval(wh_bwd)
print(f"Wh_backward[0,:5]: {wh_bwd[0, :5].tolist()}")

Wx_backward[0,:5]: [0.07535813003778458, 0.015768224373459816, 0.038377407938241
96, -0.05646621435880661, 0.059998419135808945]
Combined backward bias[:5]: [0.008811943233013153, 0.07471653074026108, 0.241935
8789920807, -0.18136098980903625, 0.07517142593860626]
Wh_forward[0,:5]: [-0.00253816950134933, -0.05682491883635521, -0.08022928237915
039, 0.1669272631406784, -0.04581911861896515]
Wh_backward[0,:5]: [0.023976480588316917, -0.13653622567653656, -0.0113054495304
82292, -0.0346134789288044, -0.08951855450868607]


In [22]:
# Quick A/B test with the LSTM recurrence fix
import subprocess, os, tempfile
import soundfile as sf

REPO_ROOT = os.path.dirname(os.getcwd())
VOICERS_CLI = os.path.join(REPO_ROOT, "target/release/voicers-cli")


def rust_gen(text, voice="af_heart"):
    wav = tempfile.mktemp(suffix=".wav")
    subprocess.run(
        [VOICERS_CLI, "--text", text, "--voice", voice, "--output", wav],
        capture_output=True,
        text=True,
        timeout=120,
    )
    return wav


test_phrases = [
    "Hello world",
    "Good morning",
    "The quick brown fox",
    "How are you doing today?",
    "She sells seashells by the seashore.",
    "I read a book yesterday.",
    "The dog is running in the park.",
]

print(f"{'Input':<45} {'Python Whisper':<40} {'Rust Whisper'}")
print("=" * 125)

for phrase in test_phrases:
    py_wav = python_generate(phrase)
    py_heard = transcribe(py_wav) if py_wav else "FAILED"
    os.unlink(py_wav) if py_wav else None

    rust_wav = rust_gen(phrase)
    rust_heard = transcribe(rust_wav)
    os.unlink(rust_wav)

    print(f"{phrase:<45} {py_heard[:38]:<40} {rust_heard[:38]}")

print("\nDone!")

Input                                         Python Whisper
       Rust Whisper


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Input                                         Python Whisper
       Rust Whisper
Hello world                                   Hello world.
       Hello world.


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                         Python Whisper
       Rust Whisper
Hello world                                   Hello world.
       Hello world.
Good morning                                  Good morning.
       Good morning.


...

/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                         Python Whisper
       Rust Whisper
Hello world                                   Hello world.
       Hello world.
Good morning                                  Good morning.
       Good morning.
The quick brown fox                           The quick brown fox.
       The quick brown fox.
How are you doing today?                      How are you doing today?
       How are you doing today?
She sells seashells by the seashore.          She sells seashells by the seashor
e.     She sells seashells by the seashore.
I read a book yesterday.                      I read a book yesterday.
       I read a book yesterday.


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                         Python Whisper
       Rust Whisper
Hello world                                   Hello world.
       Hello world.
Good morning                                  Good morning.
       Good morning.
The quick brown fox                           The quick brown fox.
       The quick brown fox.
How are you doing today?                      How are you doing today?
       How are you doing today?
She sells seashells by the seashore.          She sells seashells by the seashor
e.     She sells seashells by the seashore.
I read a book yesterday.                      I read a book yesterday.
       I read a book yesterday.
The dog is running in the park.               The dog is running in the park.
       The dog is running in the park.

Done!


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u